# Phase III — Modular Baseline Validation (Prompts 08–10)

**Do not run the 48-output main experiment yet.** This notebook checks the environment, validates the frozen dataset and prompt assembly, pins a model revision, and runs one development-only source to verify the end-to-end inference/output pipeline. The development output is excluded from the main dataset.

Run cells in order. Send the generated `phase3_baseline_validation_output.json` to the assistant for source-grounded review. Do not manually score the output.

In [ ]:
# Module 1 — Environment diagnostics and run log
import os, json, csv, gc, time, hashlib, platform, sys
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
import accelerate
import huggingface_hub

runtime_log = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "huggingface_hub_version": huggingface_hub.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "python_executable": sys.executable,
}
print(json.dumps(runtime_log, indent=2))
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU, reconnect, and rerun this cell.")

In [ ]:
# Module 2 — Upload and validate the frozen main dataset without generating outputs
from google.colab import files
DATA_FILE = "study_sources_v1_1_verified.json"
if not Path(DATA_FILE).exists():
    uploaded = files.upload()
    if DATA_FILE not in uploaded:
        raise FileNotFoundError(f"Please upload {DATA_FILE} from the experiment package.")

raw_bytes = Path(DATA_FILE).read_bytes()
dataset_sha256 = hashlib.sha256(raw_bytes).hexdigest()
dataset = json.loads(raw_bytes.decode("utf-8"))
sources = dataset.get("sources")
assert dataset.get("dataset_version") == "v1.1", f"Unexpected dataset version: {dataset.get('dataset_version')}"
assert isinstance(sources, list) and len(sources) == 8, f"Expected 8 sources, found {len(sources) if isinstance(sources, list) else type(sources)}"
required = {"id", "title", "url", "abstract", "doi", "license"}
ids = []
abstract_hashes = []
for item in sources:
    missing = required - set(item)
    assert not missing, f"{item.get('id')} missing fields: {sorted(missing)}"
    assert item["abstract"].strip(), f"Empty abstract: {item['id']}"
    assert item["url"].startswith("https://"), f"Unexpected URL: {item['id']}"
    ids.append(item["id"])
    abstract_hashes.append(hashlib.sha256(item["abstract"].encode("utf-8")).hexdigest())
assert len(set(ids)) == len(ids), "Duplicate source IDs"
assert len(set(abstract_hashes)) == len(abstract_hashes), "Duplicate abstract text detected"

dataset_report = {
    "dataset_version": dataset["dataset_version"],
    "sha256": dataset_sha256,
    "source_count": len(sources),
    "source_ids": ids,
    "all_required_fields_present": True,
    "unique_ids": True,
    "unique_abstracts": True,
}
print(json.dumps(dataset_report, indent=2))

In [ ]:
# Module 3 — Define the prompt baseline and verify prompt construction
TASK_PARAPHRASES = {
    "P0_reference": "Summarize the scientific abstract provided below.",
    "P1_paraphrase": "Provide a concise summary of the scientific abstract that follows.",
    "P2_paraphrase": "Give a concise account of the research abstract below.",
}
COMMON_INSTRUCTIONS = (
    " Use only information explicitly supported by the abstract. Do not add external knowledge, "
    "infer unstated facts, or invent quantities. Preserve the study's scope and qualifications.\n\n"
    "Return exactly these fields:\n"
    "Central conclusion: exactly one sentence stating the paper's main conclusion.\n"
    "Key findings: exactly two bullet points reporting important findings from the abstract.\n\n"
    "ABSTRACT:\n{abstract}"
)

def build_prompt(prompt_id, abstract):
    if prompt_id not in TASK_PARAPHRASES:
        raise KeyError(f"Unknown prompt condition: {prompt_id}")
    return TASK_PARAPHRASES[prompt_id] + COMMON_INSTRUCTIONS.format(abstract=abstract)

# Unit/sanity tests: same source and common instruction; all output constraints identical.
test_abstract = "A test study found an improvement. The authors noted the sample was small."
assembled = {pid: build_prompt(pid, test_abstract) for pid in TASK_PARAPHRASES}
assert len(assembled) == 3
assert all(test_abstract in p for p in assembled.values())
assert all("exactly one sentence" in p and "exactly two bullet points" in p for p in assembled.values())
assert all(COMMON_INSTRUCTIONS.format(abstract=test_abstract) in p for p in assembled.values())
print("Prompt assembly tests: PASSED")
for pid, prompt in assembled.items():
    print(pid, "| task phrase:", prompt.splitlines()[0])

In [ ]:
# Module 4 — Resolve and pin one model revision before loading it
# This cell needs Hugging Face Hub access. It fails safely if the revision cannot be resolved.
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_SHA = HfApi().model_info(MODEL_ID).sha
if not MODEL_SHA:
    raise RuntimeError("Could not resolve model commit SHA; stop before generating baseline output.")
print("Pinned model:", MODEL_ID)
print("Pinned commit SHA:", MODEL_SHA)

In [ ]:
# Module 5 — Load the pinned baseline model and test model/tokenizer compatibility
baseline_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA)
baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_SHA, dtype=torch.float16, device_map="auto"
)
baseline_model.eval()
print("Model load: PASSED")
print("Model device:", baseline_model.device)
print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / (1024**3), 3))

In [ ]:
# Module 6 — One development-only end-to-end baseline inference
# This published abstract is NOT one of the eight main-study sources and will be excluded from main results.
DEV_SOURCE = {
    "id": "DEV_STEM_2023",
    "title": "An AI-based intervention for improving undergraduate STEM learning",
    "url": "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0288844",
    "doi": "10.1371/journal.pone.0288844",
    "license": "CC BY; cite authors and source",
    "abstract": (
        "We present results from a small-scale randomized controlled trial that evaluates the impact of just-in-time "
        "interventions on the academic outcomes of N = 65 undergraduate students in a STEM course. Intervention messaging "
        "content was based on machine learning forecasting models of data collected from 537 students in the same course "
        "over the preceding 3 years. Trial results show that the intervention produced a statistically significant increase "
        "in the proportion of students that achieved a passing grade. The outcomes point to the potential and promise of "
        "just-in-time interventions for STEM learning and the need for larger fully-powered randomized controlled trials."
    )
}

prompt_id = "P0_reference"
exact_prompt = build_prompt(prompt_id, DEV_SOURCE["abstract"])
messages = [{"role": "user", "content": exact_prompt}]
chat_text = baseline_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = baseline_tokenizer(chat_text, return_tensors="pt").to(baseline_model.device)
MAX_NEW_TOKENS = 180
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
start = time.perf_counter()
with torch.inference_mode():
    generated = baseline_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=baseline_tokenizer.eos_token_id,
        eos_token_id=baseline_tokenizer.eos_token_id
    )
elapsed = time.perf_counter() - start
n_input = inputs["input_ids"].shape[1]
new_tokens = generated[0][n_input:]
raw_output = baseline_tokenizer.decode(new_tokens, skip_special_tokens=True)
assert raw_output.strip(), "Generated output is empty"

baseline_record = {
    "stage": "Phase III baseline validation; development only; excluded from main dataset",
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "runtime": runtime_log,
    "main_dataset_sha256": dataset_sha256,
    "source": DEV_SOURCE,
    "model_id": MODEL_ID,
    "model_revision_sha": MODEL_SHA,
    "prompt_id": prompt_id,
    "exact_prompt": exact_prompt,
    "raw_output": raw_output,
    "input_tokens": n_input,
    "output_tokens": len(new_tokens),
    "max_new_tokens": MAX_NEW_TOKENS,
    "possible_truncation": len(new_tokens) >= MAX_NEW_TOKENS,
    "do_sample": False,
    "seed": SEED,
    "generation_seconds": round(elapsed, 3),
}
print("Development baseline output:\n")
print(raw_output)
print("\nOutput non-empty: PASSED")
print("Possible truncation:", baseline_record["possible_truncation"])

In [ ]:
# Module 7 — Save baseline evidence and share it for source-grounded validation
BASELINE_FILE = "phase3_baseline_validation_output.json"
with open(BASELINE_FILE, "w", encoding="utf-8") as f:
    json.dump(baseline_record, f, indent=2, ensure_ascii=False)
print("Saved:", BASELINE_FILE)
from google.colab import files
files.download(BASELINE_FILE)

# Release GPU memory after output and metadata are captured.
del baseline_model, baseline_tokenizer
gc.collect()
torch.cuda.empty_cache()

## Stop here and send the JSON

Upload `phase3_baseline_validation_output.json` in the chat. Do not run the 48-output main experiment notebook yet. The next check will examine source-grounding, metadata, prompt/output integrity and possible unsupported additions. The baseline report will then be completed honestly with VERIFIED / UNCERTAIN / NEEDS CORRECTION items.